In [ ]:
import os, sys

import os; os.environ["ACCELERATE_DISABLE_RICH"] = "1"
from IPython import get_ipython
ipython = get_ipython()
ipython.run_line_magic("load_ext", "autoreload")
ipython.run_line_magic("autoreload", "2")

import torch
from torch import Tensor
from jaxtyping import Float
import itertools
from tqdm import tqdm

from transformer_lens import utils, HookedTransformer
import plotly.express as px
from functools import partial

torch.set_grad_enabled(False)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)


from utils.path_patching import Node, IterNode, path_patch, act_patch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

from transformers import PreTrainedTokenizerFast, AutoTokenizer
import transformer_lens as tl
from transformer_lens import HookedTransformer, HookedTransformerConfig
import json

from utils.plot_head import imshow

def line(tensor, **kwargs):
    px.line(
        y=utils.to_numpy(tensor),
        **kwargs,
    ).show()

%load_ext autoreload
%autoreload 2

In [ ]:
TOKENIZER_DIR  = "../model/wordlevel_tokenizer"

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, add_bos_token=True)

# --- Load config ---
with open("../model/trained_transformerlens_model/config_2l.json", "r") as f:
    cfg_dict = json.load(f)

# --- Fix dtype string back to actual torch dtype ---
if isinstance(cfg_dict.get("dtype"), str):
    cfg_dict["dtype"] = getattr(torch, cfg_dict["dtype"].replace("torch.", ""))

# --- Rebuild config and model ---
config = HookedTransformerConfig.from_dict(cfg_dict)
model = HookedTransformer(config)
model.load_state_dict(torch.load("../model/trained_transformerlens_model/model_weights_2l.pth"))
model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

model.set_tokenizer(tokenizer)

model.cfg.default_prepend_bos, model.cfg.tokenizer_prepends_bos
model.set_use_split_qkv_input(True)

In [ ]:
from data.succession import generate_successor_pairs, create_prompt, create_flipped_prompt, create_augmented_prompts, to_json_format

succession_dataset, succession_mapping = generate_successor_pairs()

# Convert dataset to prompts
task_prompts = create_prompt(succession_dataset)
flipped_task_prompts = create_flipped_prompt(succession_dataset)

aug_data = create_augmented_prompts(model.tokenizer, task_prompts, flipped_task_prompts, truncate=False, which_task='next')

create_single_prompt_lambda = lambda task_prompts: " ".join(task_prompts.values())
prompts = create_single_prompt_lambda(task_prompts)
flipped_prompts = create_single_prompt_lambda(flipped_task_prompts)

answer_words = []
for task_name in task_prompts:
    # Split prompt into elements
    prompt_elems = task_prompts[task_name].split()
    flipped_elems = flipped_task_prompts[task_name].split()
    # Remove last element and store as answer
    answer_not_flipped = prompt_elems.pop(-1)
    answer_flipped = flipped_elems.pop(-1)
    answer_words.append((answer_not_flipped, answer_flipped))
    # Update prompts without last element
    task_prompts[task_name] = " ".join(prompt_elems)
    flipped_task_prompts[task_name] = " ".join(flipped_elems)

print("Before flipping:")
clean_prompts = []
for task_name, task_prompt in task_prompts.items():
    print(f"{task_name}: {task_prompt}")
    clean_prompts.append(task_prompt)


print("\nAfter flipping:")
flipped_prompts = []
for task_name, task_prompt in flipped_task_prompts.items():
    print(f"{task_name}: {task_prompt}")
    flipped_prompts.append(task_prompt)

print("\nAnswers (not_flipped, flipped):")
single_tok_answers = []
correct_tok_answers = []
wrong_tok_answers = []
for correct, wrong in answer_words:
    # encode without BOS and without padding
    tok_corr = model.to_tokens([correct], prepend_bos=False).tolist()[0]
    tok_wrong = model.to_tokens([wrong], prepend_bos=False).tolist()[0]
    # strip off any padding or BOS; keep only the “real” token IDs
    # (you know your seq‐len so you can take the non-zero part)
    # but simplest is to see if each length == 1:
    # if len(tok_corr) == 1 and len(tok_wrong) == 1:
    single_tok_answers.append((tok_corr[0], tok_wrong[0]))
    correct_tok_answers.append(tok_corr[0])
    wrong_tok_answers.append(tok_wrong[0])
    print(f"Answer: {correct} ({model.to_string(tok_corr[0])}) vs {wrong} ({model.to_string(tok_wrong[0])})")
# answers = [ans[::i] for ans in single_tok_answers for i in (1, -1)]
# print(answers)

correct_tok_answers = torch.tensor(correct_tok_answers, dtype=torch.long)
correct_tok_answers = correct_tok_answers.to(device)

wrong_tok_answers = torch.tensor(wrong_tok_answers, dtype=torch.long)
wrong_tok_answers = wrong_tok_answers.to(device)

answer_tokens = torch.tensor(single_tok_answers, dtype=torch.long)
answer_tokens = answer_tokens.to(device)

In [ ]:
aug_clean_prompts = [aug["clean"] for aug in aug_data]
aug_flipped_prompts = [aug["corrupt"] for aug in aug_data]
for i, (task_name, task_prompt) in enumerate(zip(task_prompts, flipped_task_prompts)):
    aug_prompt = aug_data[i]["clean"]
    print(f"Task: {task_name}, Augmented Prompt: {aug_prompt}")
    print(f"Length of toks: {len(model.to_str_tokens(aug_prompt))}")

In [ ]:
print("Clean prompts:")
for i, (_, task_prompt) in enumerate(task_prompts.items()):
    utils.test_prompt(task_prompt, answer_words[i][0], model, device, print_details=False)
print("Flipped prompts:")
for i, (_, task_prompt) in enumerate(flipped_task_prompts.items()):
    utils.test_prompt(task_prompt, answer_words[i][1], model, device, print_details=False)

In [ ]:
from utils.direct_logit_attribution import logits_to_ave_logit_diff

clean_tokens = model.to_tokens(aug_clean_prompts, prepend_bos=False, padding_side='left').to(device)
flipped_tokens = model.to_tokens(aug_flipped_prompts, prepend_bos=False, padding_side='left').to(device)

clean_logits, clean_cache = model.run_with_cache(clean_tokens)
flipped_logits, flipped_cache = model.run_with_cache(flipped_tokens)

clean_logit_diff = logits_to_ave_logit_diff(clean_logits, answer_tokens)
flipped_logit_diff = logits_to_ave_logit_diff(flipped_logits, answer_tokens)

print(
    "Clean string 0:    ", model.to_string(clean_tokens[0]), "\n"
    "Flipped string 0:", model.to_string(flipped_tokens[0])
)
print(f"Clean logit diff: {clean_logit_diff:.4f}")
print(f"Flipped logit diff: {flipped_logit_diff:.4f}")

def pp_metric_denoising(
    logits: Float[Tensor, "batch seq d_vocab"],
    answer_tokens: Float[Tensor, "batch 2"] = answer_tokens,
    flipped_logit_diff: float = flipped_logit_diff,
    clean_logit_diff: float = clean_logit_diff,
) -> Float[Tensor, ""]:
    '''
    Linear function of logit diff, calibrated so that it equals 0 when performance is
    same as on flipped input, and 1 when performance is same as on clean input.
    '''
    patched_logit_diff = logits_to_ave_logit_diff(logits, answer_tokens)
    return ((patched_logit_diff - flipped_logit_diff) / (clean_logit_diff  - flipped_logit_diff)).item()

labels = [f"{tok} {i}" for i, tok in enumerate(model.to_str_tokens(clean_tokens[0]))]

In [ ]:
aug_prompt = f"the next term in the sequence {task_prompts['Numbers']} is"
aug_toks = model.to_tokens(aug_prompt, prepend_bos=False, padding_side='left').to(device)

# Run the model with the augmented prompt
aug_logits, aug_cache = model.run_with_cache(aug_toks)
print(f"Augmented prompt: {aug_prompt}",f"Answer toks are: {model.to_str_tokens(answer_tokens[0])}")

In [ ]:
import numpy as np
from utils.direct_logit_attribution import logit_lens

logit_lens_logit_diffs, labels = logit_lens(model, aug_clean_prompts, clean_cache, answer_tokens, decomposition="residual")

line(
    logit_lens_logit_diffs,
    x=np.arange(model.cfg.n_layers * 2 + 1) / 2,
    hover_name=labels,
    labels={"x": "Layer", "y": "Logit Diff"},
    title="Logit Difference From Accumulate Residual Stream",
)

In [ ]:
per_layer_logit_diffs, labels = logit_lens(model, aug_clean_prompts, clean_cache,  answer_tokens, decomposition="layer_blocks")

per_layer_logit_diffs = per_layer_logit_diffs.cpu().numpy()  # Move tensor to CPU and convert to NumPy array
line(per_layer_logit_diffs, hover_name=labels, title="Logit Difference From Each Layer", labels={"x": "Layer Activation (Attention/MLP)", "y": "Logit Diff"})

In [ ]:
per_head_logit_diffs, labels = logit_lens(model, aug_clean_prompts, clean_cache, answer_tokens, decomposition="attention_heads")

per_head_logit_diffs_np = per_head_logit_diffs.cpu().numpy()
imshow(
    per_head_logit_diffs_np,
    labels={"x": "Head", "y": "Layer"},
    title="Logit Difference From Each Head",
    width=800,
)

In [ ]:
try: 
    from circuitsvis.attention import attention_heads
except ImportError:
    print("circuitsvis not installed. Installing...")
    # Install circuitsvis if not already installed
    %pip install circuitsvis==1.43.3
    # Retry importing after installation
    from circuitsvis.attention import attention_heads

from typing import List, Optional, Union
from transformer_lens import ActivationCache
from IPython.display import HTML

def visualize_attention_patterns(
    heads: Union[List[int], int, Float[torch.Tensor, "heads"]],
    local_cache: ActivationCache,
    local_tokens: torch.Tensor,
    title: Optional[str] = "",
    max_width: Optional[int] = 700,
) -> str:
    # If a single head is given, convert to a list
    if isinstance(heads, int):
        heads = [heads]

    # Create the plotting data
    labels: List[str] = []
    patterns: List[Float[torch.Tensor, "dest_pos src_pos"]] = []

    # Assume we have a single batch item
    batch_index = 0

    for head in heads:
        # Set the label
        layer = head // model.cfg.n_heads
        head_index = head % model.cfg.n_heads
        labels.append(f"L{layer}H{head_index}")

        # Get the attention patterns for the head
        # Attention patterns have shape [batch, head_index, query_pos, key_pos]
        patterns.append(local_cache["attn", layer][batch_index, head_index])

    # Convert the tokens to strings (for the axis labels)
    str_tokens = model.to_str_tokens(local_tokens)

    # Combine the patterns into a single tensor
    patterns: Float[torch.Tensor, "head_index dest_pos src_pos"] = torch.stack(
        patterns, dim=0
    )

    # Circuitsvis Plot (note we get the code version so we can concatenate with the title)
    plot = attention_heads(
        attention=patterns, tokens=str_tokens, attention_head_names=labels
    ).show_code()

    # Display the title
    title_html = f"<h2>{title}</h2><br/>"

    # Return the visualisation as raw code
    return f"<div style='max-width: {str(max_width)}px;'>{title_html + plot}</div>"

top_k = 3

top_positive_logit_attr_heads = torch.topk(
    per_head_logit_diffs.flatten(), k=top_k
).indices

positive_html = visualize_attention_patterns(
    top_positive_logit_attr_heads,
    aug_cache,
    aug_toks,
    f"Top {top_k} Positive Logit Attribution Heads",
)

top_negative_logit_attr_heads = torch.topk(
    -per_head_logit_diffs.flatten(), k=top_k
).indices

negative_html = visualize_attention_patterns(
    top_negative_logit_attr_heads,
    aug_cache,
    aug_toks,
    title=f"Top {top_k} Negative Logit Attribution Heads",
)

HTML(positive_html + negative_html)

In [ ]:
# from utils.detect_head import detect_head, get_supported_heads
# from utils.plot_head import imshow

# aug_prompt = f"the next term in the sequence {task_prompts['Numbers']} is"
# utils.test_prompt(aug_prompt, answer="20", model=model, prepend_space_to_answer=True, print_details=False)
# print(aug_prompt)
# number_toks = model.to_tokens(aug_prompt, prepend_bos=False)

# model.reset_hooks()
# _, cache = model.run_with_cache(
#     # text[:100],
#     number_toks,
#     )

# get_supported_heads()

# last_successor_heads_scores = detect_head(model, seq=aug_prompt, detection_pattern='last_successor_head', cache=None, error_measure='abs')

# def plot_head_detection_scores(
#     scores: torch.Tensor,
#     zmin: float = -1,
#     zmax: float = 1,
#     xaxis: str = "Head",
#     yaxis: str = "Layer",
#     title: str = "Head Matches"
# ) -> None:
#     imshow(scores, zmin=zmin, zmax=zmax, xaxis=xaxis, yaxis=yaxis, title=title)

# plot_head_detection_scores(last_successor_heads_scores, title="Last Successor Head Matches")

## Path Patching

In [ ]:
def _pp_metric_noising(
        logits: Float[Tensor, "batch seq d_vocab"],
        clean_logit_diff: float,
        corrupted_logit_diff: float,
        answer_tokens: Float[Tensor, "batch 2"] = answer_tokens,
    ) -> float:
        '''
        We calibrate this so that the value is 0 when performance isn't harmed (i.e. same as IOI dataset),
        and -1 when performance has been destroyed (i.e. is same as ABC dataset).
        '''
        patched_logit_diff = logits_to_ave_logit_diff(logits, answer_tokens)
        return ((patched_logit_diff - clean_logit_diff) / (clean_logit_diff - corrupted_logit_diff)).item()

def generate_data_and_caches(verbose: bool = False):

    model.reset_hooks(including_permanent=True)

    abc_logits_original, abc_cache = model.run_with_cache(clean_tokens.long())
    cba_logits_original, cba_cache = model.run_with_cache(flipped_tokens.long())

    abc_average_logit_diff = logits_to_ave_logit_diff(abc_logits_original, answer_tokens).item()
    cba_average_logit_diff = logits_to_ave_logit_diff(cba_logits_original, answer_tokens).item()

    if verbose:
        print(f"Average logit diff (ABC dataset): {abc_average_logit_diff:.4f}")
        print(f"Average logit diff (CBA dataset): {cba_average_logit_diff:.4f}")

    pp_metric_noising = partial(
        _pp_metric_noising,
        clean_logit_diff=abc_average_logit_diff,
        corrupted_logit_diff=cba_average_logit_diff,
        answer_tokens=answer_tokens,
    )

    return abc_cache, cba_cache, pp_metric_noising

abc_cache, cba_cache, pp_metric_noising = generate_data_and_caches(verbose=True)

In [ ]:
results_direct = path_patch(
    model,
    orig_input=clean_tokens,
    new_input=flipped_tokens,
    sender_nodes=IterNode('z'), # This means iterate over all heads in all layers
    receiver_nodes=Node('resid_post', 1), # This is final resid_post 
    patching_metric=pp_metric_noising,
    direct_includes_mlps=False,
    verbose=True
)

imshow(
    torch.stack([results_direct['z'], results_direct['z']]),
    facet_col=0, facet_labels=["Direct path includes MLPs", "Direct path is just skip connections"],
    title="Each attention head's direct effect on logit difference",
    labels={"x": "Head", "y": "Layer", "color": "Logit diff variation"},
    # border=True,
    width=950,
    # margin={"r": 100, "l": 100}
)

In [ ]:
MOVERS = [(0, 1), (0, 3), (0, 2), (1, 2)]

results = path_patch(
    model,
    orig_input=clean_tokens,
    new_input=flipped_tokens,
    sender_nodes=[Node("z", layer, head=head) for layer, head in MOVERS],
    receiver_nodes=Node("resid_post", 1), # This is resid_post at layer 11
    patching_metric=pp_metric_noising,
)

print(results)

In [ ]:
results = path_patch(
    model,
    orig_input=flipped_tokens,
    new_input=clean_tokens,
    sender_nodes=IterNode(["resid_pre", "attn_out", "mlp_out"], seq_pos="each"),
    receiver_nodes=Node("resid_post", 1),
    patching_metric=pp_metric_denoising,
    direct_includes_mlps=False, # gives similar results to direct_includes_mlps=True
    verbose=True,
)
# We get a dictionary where each key is a node name, and each value is a tensor of (layer, seq_pos)
assert list(results.keys()) == ['resid_pre', 'attn_out', 'mlp_out']
# assert results["resid_pre"].shape == (14, 12)

results_stacked = torch.stack([
    results.T for results in results.values()
])

# Ensure x labels match the number of sequence positions
# x_labels = [str(i) for i in range(results_stacked.shape[-1])]
x_labels = model.to_str_tokens(clean_tokens[0])

imshow(
    results_stacked,
    facet_col=0,
    facet_labels=['resid_pre', 'attn_out', 'mlp_out'],
    title="Results of denoising patching at residual stream",
    labels={"x": "Sequence position", "y": "Layer", "color": "Logit diff variation"},
    x=x_labels,
    # xaxis_tickangle=45,
    width=1300,
    # margin={"r": 100, "l": 100},
    # border=True,
)

### Head to head Patching

In [ ]:
results = path_patch(
    model,
    orig_input=clean_tokens,
    new_input=flipped_tokens,
    sender_nodes=IterNode("z"),
    receiver_nodes=[Node("q", layer, head=head) for layer, head in MOVERS], # [(1, 2)]
    patching_metric=pp_metric_noising,
    verbose=True,
)

imshow(
    results['z'][:9] * 100,
    title="Direct effect on Name Mover Heads' queries",
    labels={"x": "Head", "y": "Layer", "color": "Logit diff variation"},
    # coloraxis=dict(colorbar_ticksuffix = "%"),
    # border=True,
    width=700,
)

In [ ]:
results = path_patch(
    model,
    orig_input=clean_tokens,
    new_input=flipped_tokens,
    sender_nodes=IterNode("z"),
    receiver_nodes=[Node("k", layer, head=head) for layer, head in MOVERS], # [(1, 2)]
    patching_metric=pp_metric_noising,
    verbose=True,
)

imshow(
    results['z'][:9] * 100,
    title="Direct effect on Name Mover Heads' keys",
    labels={"x": "Head", "y": "Layer", "color": "Logit diff variation"},
    # coloraxis=dict(colorbar_ticksuffix = "%"),
    # border=True,
    width=700,
)

In [ ]:
results = path_patch(
    model,
    orig_input=clean_tokens,
    new_input=flipped_tokens,
    sender_nodes=IterNode("z"),
    receiver_nodes=[Node("v", layer, head=head) for layer, head in MOVERS], # [(1, 2)]
    patching_metric=pp_metric_noising,
    verbose=True,
)

imshow(
    results['z'][:9] * 100,
    title="Direct effect on Name Mover Heads' values",
    labels={"x": "Head", "y": "Layer", "color": "Logit diff variation"},
    # coloraxis=dict(colorbar_ticksuffix = "%"),
    # border=True,
    width=700,
)

In [ ]:
POSITIONAL_HEADS = [(0, 1), (1, 0), (1, 1)]

results_ind_q = path_patch(
    model,
    orig_input=clean_tokens,
    new_input=flipped_tokens,
    sender_nodes=IterNode("z"),
    receiver_nodes=[Node("q", layer, head=head) for layer, head in POSITIONAL_HEADS],
    patching_metric=pp_metric_noising,
    verbose=True,
)

results_ind_k = path_patch(
    model,
    orig_input=clean_tokens,
    new_input=flipped_tokens,
    sender_nodes=IterNode("z"),
    receiver_nodes=[Node("k", layer, head=head) for layer, head in POSITIONAL_HEADS],
    patching_metric=pp_metric_noising,
    verbose=True,
)
results_ind_v = path_patch(
    model,
    orig_input=clean_tokens,
    new_input=flipped_tokens,
    sender_nodes=IterNode("z"),
    receiver_nodes=[Node("v", layer, head=head) for layer, head in POSITIONAL_HEADS],
    patching_metric=pp_metric_noising,
    verbose=True,
)

imshow(
    torch.stack([results_ind_q['z'], results_ind_k['z'], results_ind_v['z']])[:, :8] * 100,
    facet_col=0, facet_labels=["Q", "K", "V"],
    title="Direct effect on Positional Heads' queries, keys, and values",
    labels={"x": "Head", "y": "Layer", "color": "Logit diff variation"},
    # coloraxis=dict(colorbar_ticksuffix = "%"),
    # border=True,
    # width=700,
)